| **Onderdeel**       | **Inhoud** |
|----------------------|------------|
| 📘 **Titel** | Statistics for Data Science — Deel 3: Logistische Regressie (*Hoog slagingssucces > 60 %*) |
| 👤 **Auteur** | `Adil Vural` & `Oktay Doğan` |
| 🏛️ **Instelling** | Erasmus Q-Intelligence |
| 🗓️ **Datum / Versie** | 20 oktober 2025 — Versie 1.0 |
| 🧾 **Bestand** | `Opdracht_3_LogistischeRegressie_slagingssucces_final.ipynb` |
| 🧮 **Dataset** | `college_statistics.csv` |
| 🧠 **Doel** | Ontwikkelen van een logistisch regressiemodel dat de **kans voorspelt dat een universiteit een hoog slagingssucces heeft** (`slagingssucces_bin = 1 ↔ Graduation Rate > 60 %`), op basis van institutionele, academische en financiële kenmerken. |
| 🔍 **Analyse-onderdelen** | 1️⃣ Creëren van binaire doelvariabele (`slagingssucces_bin`) <br>2️⃣ Opdeling van de data in *estimation* (train) en *test* sample <br>3️⃣ Schatting van het logistische model met transformaties (`accept_rate`, `enroll_rate`, `log(Apps)`) <br>4️⃣ Controle op significantie en multicollineariteit <br>5️⃣ Backward selection voor modelvereenvoudiging <br>6️⃣ Evaluatie van modelprestaties (accuracy, recall, AUC) <br>7️⃣ Conclusies en interpretatie van significante factoren (`log(Apps)`, `Outstate`, `Top10perc`, `Private_bin`) |
| 📚 **Bronnen** | *Practical Statistics for Data Scientists* (2e editie) — Bruce, Bruce & Gedeck <br>Lecture 7 — Erasmus “Statistics for Data Science” 2025-2026 |


## Opdracht Deel 3 — Logistische Regressie

Het doel van Deel 3 is om te onderzoeken hoe logistische regressie gebruikt kan worden om de **kans te voorspellen dat een universiteit een hoog slagingssucces behaalt** wanneer de afhankelijke variabele binair is (Ja/Nee, 1/0).  
In deze opdracht passen we deze methode toe op de *College*-dataset.  
We modelleren de kans dat een universiteit een **afstudeerpercentage boven de 60 %** heeft (`slagingssucces_bin = 1`) op basis van verschillende institutionele, academische en financiële kenmerken.  

De centrale onderzoeksvraag luidt:  
**Welke factoren verklaren het beste waarom sommige universiteiten een hoog slagingssucces behalen, terwijl andere dat niet doen?**




## 6 Logistische Regressie




In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import warnings
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, roc_auc_score

***

In [9]:
# lees de dataset in een pandas DataFrame
df_college = pd.read_csv("college_statistics.csv", sep=",")
df_college.columns = [c.replace(".", "_") for c in df_college.columns]
df_college.rename(columns={'Unnamed: 0': 'University'}, inplace=True)
print (df_college.columns)

Index(['University', 'Private', 'Apps', 'Accept', 'Enroll', 'Top10perc',
       'Top25perc', 'F_Undergrad', 'P_Undergrad', 'Outstate', 'Room_Board',
       'Books', 'Personal', 'PhD', 'Terminal', 'S_F_Ratio', 'perc_alumni',
       'Expend', 'Grad_Rate'],
      dtype='object')


In [10]:
# === 6(a) Nieuwe variabele 'slagingssucces_bin' aanmaken ===

# Als de kolom bijvoorbeeld 'Grad.Rate' of 'Grad_Rate' heet:
# Maak de binaire variabele: 1 = hoger dan 60%, 0 = 60% of lager
df_college["slagingssucces_bin"] = (df_college["Grad_Rate"] > 60).astype(int)

# Controle: eerste paar rijen
print(df_college[["Grad_Rate", "slagingssucces_bin"]].head())

# Samenvatting (optioneel)
print(df_college["slagingssucces_bin"].value_counts(normalize=True))


   Grad_Rate  slagingssucces_bin
0         60                   0
1         56                   0
2         54                   0
3         59                   0
4         15                   0
slagingssucces_bin
1    0.610039
0    0.389961
Name: proportion, dtype: float64


***

(b) Deel de data opnieuw op in een estimation en een test sample (of hergebruik de
eerdere opdeling).

We verdelen de dataset in twee delen:
 - Estimation sample (trainingsset) → om het logistische model te schatten;
 - Test sample → om de voorspellende prestaties te valideren.
Dit voorkomt overfitting en maakt het mogelijk de generaliseerbaarheid van het model te meten.


In [11]:
# Vraag 6b — Data opdelen in estimation (train) en test sample
# Doel- en verklarende variabelen
# Doelvariabele (uit 6a)
y = df_college['slagingssucces_bin']

# Enkele potentiële verklarende variabelen
X = df_college[['Outstate', 'PhD', 'Top10perc', 'slagingssucces_bin', 'Apps', 'Grad_Rate']].copy()

#Splitsing: 75% train, 25% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,          # 25% test
    stratify=y,              # behoud 0/1 verhouding
    random_state=42          # reproduceerbaar
)

# Controleer de verdeling
print("Verdeling totaal :", y.value_counts(normalize=True).round(3).to_dict())
print("Verdeling train  :", y_train.value_counts(normalize=True).round(3).to_dict())
print("Verdeling test   :", y_test.value_counts(normalize=True).round(3).to_dict())


Verdeling totaal : {1: 0.61, 0: 0.39}
Verdeling train  : {1: 0.61, 0: 0.39}
Verdeling test   : {1: 0.61, 0: 0.39}


## Conclusie:

 - slagingssucces_bin = 1 → hoog slagingssucces (Grad_Rate > 60%)
 - slagingssucces_bin = 0 → laag slagingssucces (Grad_Rate ≤ 60%)
 
Uit de resultaten blijkt dat ongeveer 61% van de universiteiten een hoog slagingssucces heeft,
en 39% een laag slagingssucces.
Omdat de verdeling exact hetzelfde is in de train, test en totaal,
werkt de stratificatie (via stratify=y) perfect — er is dus geen scheefheid tussen de datasets.

De dataset is succesvol opgesplitst in een estimation sample (train) en een test sample met een verhouding van 75% / 25%.
De verdeling van de doelvariabele slagingssucces_bin is stabiel in beide subsets:
61% van de universiteiten heeft een hoog slagingssucces (> 60%), en 39% een laag slagingssucces (≤ 60%).
Hierdoor kunnen we het logistische model betrouwbaar schatten op de trainingsset en de prestaties beoordelen op de testset zonder vertekening.


***

(c) Maak met behulp van de estimation data een logit model om de slagingssucces
variabele te verklaren. Denk hierbij goed na over transformaties van je variabelen. Bijvoorbeeld heeft het zin om het aantal applicaties, aantal acceptaties,
en het aantal enrollments in hetzelfde model op te nemen? Of kunnen sommige
van deze variabelen beter als percentages opgenomen worden?

Het doel is om te schatten hoe waarschijnlijk het is dat een student (of instelling) succesvol is, op basis van beschikbare kenmerken.
hier gebruiken wij de trainingsdata (niet de testdata).

| Variabele       | Betekenis                   | Formule           | Waarom beter                      |
| --------------- | --------------------------- | ----------------- | --------------------------------- |
| **accept_ratio** | Acceptatiegraad             | `Accept / Apps`   | Meet toelatingsbeleid             |
| **enroll_ratio** | Inschrijfpercentage         | `Enroll / Accept` | Meet inschrijfgedrag na toelating |
| **Apps_log**    | Log van aantal aanmeldingen | `np.log(Apps)`    | Neemt schaalverschillen weg       |



In [12]:
# === 6(c) Logistisch model voor 'slagingssucces_bin' — zonder leakage & zonder warnings ===


# 1) bepaal ratios en transformaties in de volledige dataset
df_college["accept_ratio"] = np.where(
    df_college["Apps"] > 0, df_college["Accept"] / df_college["Apps"], np.nan
)
df_college["enroll_ratio"] = np.where(
    df_college["Accept"] > 0, df_college["Enroll"] / df_college["Accept"], np.nan
)

# Clip ratio's om numerieke instabiliteit te vermijden
#Betekenis:
#Alle waarden < 0 worden 0
#Alle waarden > 1 worden 1
#Waarden tussen 0 en 1 blijven onveranderd

df_college["accept_ratio"] = df_college["accept_ratio"].clip(0, 1)
df_college["enroll_ratio"] = df_college["enroll_ratio"].clip(0, 1)



# Stabiele log-transformatie voor schaal
df_college["log_apps"] = np.log1p(df_college["Apps"])

# Optioneel: Private -> binaire indicator als kolom bestaat
if "Private" in df_college.columns:
    df_college["Private_bin"] = (
        df_college["Private"].astype(str).str.strip().str.lower()
        .isin(["yes", "true", "private", "1"])
    ).astype(int)

# 2) Verklarende variabelen (GEEN Grad_Rate en GEEN slagingssucces_bin in X!)
verklarende_variabelen = [
    "accept_ratio",
    "enroll_ratio",
    "log_apps",
    "Outstate",
    "PhD",
    "Top10perc",
    "Private_bin"
]

# 3) Align met jouw train/test-split uit 6(b)
X_train_raw = df_college.loc[X_train.index, verklarende_variabelen].copy()
X_test_raw  = df_college.loc[X_test.index,  verklarende_variabelen].copy()
y_train_sel = y_train  # uit 6(b)

# 4) Imputatie met TRAIN-statistieken (voorkomt lekken uit test)
train_medians = X_train_raw.median(numeric_only=True)
X_train_imp = X_train_raw.fillna(train_medians)
X_test_imp  = X_test_raw.fillna(train_medians)

# 5) Schalen (alleen continue kolommen; binaries ongemoeid laten)
cont_cols = []
for c in X_train_imp.columns:
    vals = pd.unique(X_train_imp[c].dropna())
    if len(vals) > 2 or not set(vals).issubset({0, 1}):
        cont_cols.append(c)

scaler = StandardScaler()
X_train_imp[cont_cols] = scaler.fit_transform(X_train_imp[cont_cols])
X_test_imp[cont_cols]  = scaler.transform(X_test_imp[cont_cols])

# 6) Intercept
X_train_sm = sm.add_constant(X_train_imp, has_constant="add")

# 7) Logit fit — eerst MLE; bij separatie/overflow -> ridge-regularized fallback
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=RuntimeWarning)
    warnings.filterwarnings("ignore", category=sm.tools.sm_exceptions.PerfectSeparationWarning)
    try:
        logit_fit = sm.Logit(y_train_sel, X_train_sm).fit(disp=False, maxiter=200)
        fit_used = "MLE (gewone Logit)"
    except Exception:
        # Ridge (L2) regularisatie stabiliseert bij (quasi) perfecte separatie
        logit_fit = sm.Logit(y_train_sel, X_train_sm).fit_regularized(
            method="l1", alpha=1.0, L1_wt=0.0, disp=False
        )
        fit_used = "Ridge-regularized Logit (fit_regularized)"

print(f"=== Logit — {fit_used} ===")
try:
    print(logit_fit.summary())
except Exception:
    print(pd.Series(logit_fit.params, name="coef").to_frame())

# 8) Odds ratio's (+ 95% CI als beschikbaar)
params = logit_fit.params
try:
    conf = logit_fit.conf_int()
    or_table = pd.DataFrame({
        "OR": np.exp(params),
        "CI_low": np.exp(conf[0]),
        "CI_high": np.exp(conf[1]),
    })
except Exception:
    or_table = pd.DataFrame({"OR": np.exp(params)})
print("\n=== Odds Ratio's (logit) ===")
print(or_table.round(3))


=== Logit — MLE (gewone Logit) ===
                           Logit Regression Results                           
Dep. Variable:     slagingssucces_bin   No. Observations:                  582
Model:                          Logit   Df Residuals:                      574
Method:                           MLE   Df Model:                            7
Date:                Thu, 23 Oct 2025   Pseudo R-squ.:                  0.2696
Time:                        15:14:04   Log-Likelihood:                -284.28
converged:                       True   LL-Null:                       -389.22
Covariance Type:            nonrobust   LLR p-value:                 9.212e-42
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
const            0.0513      0.302      0.170      0.865      -0.541       0.643
accept_ratio    -0.0470      0.137     -0.343      0.732      -0.316       0.222
enroll_ra

## Conclusie :

Het logistische model verklaart ongeveer 27 % van de variantie in het slagingssucces van universiteiten en is statistisch zeer significant (p < 0.001).
De belangrijkste voorspellers van een hoog slagingssucces zijn:
het aantal aanmeldingen (log_apps),
de hoogte van het out-of-state collegegeld (Outstate),
de kwaliteit van de instroom (Top10perc),
en de instellingstype-dummy (Private_bin).
Deze variabelen tonen dat universiteiten die meer en betere studenten aantrekken, hogere collegegelden hanteren en privé zijn, een aanzienlijk grotere kans op hoog slagingssucces hebben.
De variabelen accept_ratio en PhD blijken geen significante invloed te hebben binnen dit model.


***

(d) Gebruik wederom backward selection om het aantal verklarende variabelen te
verkleinen.

Backward selection begint met alle variabelen in het model en verwijdert vervolgens stapsgewijs de minst significante (hoogste p-waarde),
totdat alleen nog significante verklarende variabelen overblijven (p < 0.05 of 0.10 afhankelijk van criterium).

In [13]:
# === 6(d) Backward Selection (logistische regressie) ===

# ---------- 0) Zorg dat er een train/test split is ----------
if "X_train" not in locals() or "y_train" not in locals():
    # minimale fallback (mocht 6b nog niet gedraaid zijn)
    y = df_college["slagingssucces_bin"]
    X_min = df_college[["Outstate", "PhD", "Top10perc", "Apps"]].copy()
    X_train, X_test, y_train, y_test = train_test_split(
        X_min, y, test_size=0.25, random_state=42, stratify=y
    )

# ---------- 1) Features zoals in 6(c) ----------
df_college["accept_ratio]"] = np.where(
    df_college["Apps"] > 0, df_college["Accept"] / df_college["Apps"], np.nan
).clip(0, 1)
df_college["enroll_ratio"] = np.where(
    df_college["Accept"] > 0, df_college["Enroll"] / df_college["Accept"], np.nan
).clip(0, 1)
df_college["log_apps"] = np.log1p(df_college["Apps"])

if "Private" in df_college.columns:
    df_college["Private_bin"] = (
        df_college["Private"].astype(str).str.strip().str.lower()
        .isin(["yes", "true", "private", "1"]).astype(int)
    )

predictors = ["accept_ratio", "enroll_ratio", "log_apps", "Outstate", "PhD", "Top10perc","Private_bin"]

# Align met je split
X_train_raw = df_college.loc[X_train.index, predictors].copy()
y_train_sel = y_train.copy()

# Imputatie met TRAIN-median
train_medians = X_train_raw.median(numeric_only=True)
X_train_imp = X_train_raw.fillna(train_medians)

# Schalen (alleen continue kolommen)
cont_cols = []
for c in X_train_imp.columns:
    vals = pd.unique(X_train_imp[c].dropna())
    if len(vals) > 2 or not set(vals).issubset({0,1}):
        cont_cols.append(c)
scaler = StandardScaler()
X_train_imp[cont_cols] = scaler.fit_transform(X_train_imp[cont_cols])

# Voeg intercept toe
X_train_sm = sm.add_constant(X_train_imp, has_constant="add")

# ---------- 2) Helpers voor selectie ----------
def fit_logit_safe(X, y):
    """Probeer MLE; raise bij separatie/overflow zodat we fallback kunnen doen."""
    with warnings.catch_warnings():
        warnings.filterwarnings("error", category=RuntimeWarning)
        warnings.filterwarnings("error", category=sm.tools.sm_exceptions.PerfectSeparationWarning)
        res = sm.Logit(y, X).fit(disp=False, maxiter=200)
    return res

def aic_of_model(res):
    # AIC = -2*llf + 2*k
    return -2*res.llf + 2*res.df_modelwc

def backward_by_pvalue(X, y, p_thresh=0.05, max_steps=25):
    cols = list(X.columns)
    if "const" not in cols:
        X = sm.add_constant(X, has_constant="add")
        cols = list(X.columns)
    for _ in range(max_steps):
        res = fit_logit_safe(X[cols], y)
        pvals = res.pvalues.drop("const", errors="ignore")
        worst = pvals.idxmax()
        worst_p = pvals.max()
        if worst_p <= p_thresh:
            return res, cols, "pvalue"
        cols.remove(worst)
    # safety return
    res = fit_logit_safe(X[cols], y)
    return res, cols, "pvalue"

def backward_by_aic(X, y, max_steps=25, tol=1e-6):
    cols = list(X.columns)
    if "const" not in cols:
        X = sm.add_constant(X, has_constant="add")
        cols = list(X.columns)
    # init
    best_res = fit_logit_safe(X[cols], y)
    best_aic = aic_of_model(best_res)
    improved = True
    steps = 0
    while improved and steps < max_steps and len(cols) > 2:  # minstens const + 1 predictor
        improved = False
        candidate_removal = None
        candidate_res = None
        for c in [c for c in cols if c != "const"]:
            trial_cols = [k for k in cols if k != c]
            try:
                res_try = fit_logit_safe(X[trial_cols], y)
                aic_try = aic_of_model(res_try)
            except Exception:
                continue  # sla onstabiele fits over
            if aic_try + tol < best_aic:
                best_aic = aic_try
                candidate_removal = c
                candidate_res = res_try
                improved = True
        if improved and candidate_removal is not None:
            cols.remove(candidate_removal)
            best_res = candidate_res
            steps += 1
    return best_res, cols, "aic"

# ---------- 3) Run backward selection ----------
try:
    final_res, final_cols, method = backward_by_pvalue(X_train_sm, y_train_sel, p_thresh=0.05)
except Exception:
    # separatie/overflow → gebruik AIC-criterium
    final_res, final_cols, method = backward_by_aic(X_train_sm, y_train_sel)

print("=== 6(d) Backward Selection — resultaat ===")
print(f"Methode: {'p-waarde (0.05-drempel)' if method=='pvalue' else 'AIC (fallback bij separatie)'}")
print(f"Overgebleven predictoren: {final_cols}")

try:
    print(final_res.summary())
except Exception:
    # Mocht summary niet beschikbaar zijn (zou zelden gebeuren bij MLE)
    print(pd.Series(final_res.params, name="coef").to_frame())

# (optioneel) Odds ratio's tonen
or_table = pd.DataFrame({
    "OR": np.exp(final_res.params),
})
print("\nOdds Ratio's (finale model):")
print(or_table.round(3))


=== 6(d) Backward Selection — resultaat ===
Methode: p-waarde (0.05-drempel)
Overgebleven predictoren: ['const', 'log_apps', 'Outstate', 'Top10perc', 'Private_bin']
                           Logit Regression Results                           
Dep. Variable:     slagingssucces_bin   No. Observations:                  582
Model:                          Logit   Df Residuals:                      577
Method:                           MLE   Df Model:                            4
Date:                Thu, 23 Oct 2025   Pseudo R-squ.:                  0.2646
Time:                        15:14:04   Log-Likelihood:                -286.24
converged:                       True   LL-Null:                       -389.22
Covariance Type:            nonrobust   LLR p-value:                 1.966e-43
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
const           0.0446      0.292      0.15

## Conclusie:

Het vereenvoudigde logistische model na backward selection laat zien dat slechts vier factoren significant bijdragen aan een hoog slagingssucces van universiteiten:
 - (1) het aantal aanmeldingen (log (Apps)),
 - (2) het out-of-state collegegeld,
 - (3) het aandeel top-10 % studenten, en
 - (4) de private status van de instelling.
Deze variabelen verhogen elk afzonderlijk de kans op een hoge slagingsgraad, waarbij private universiteiten en instellingen met hogere instroomkwaliteit het meest kansrijk zijn.
Het model behoudt ruim een kwart van de verklaarde variantie (Pseudo R² ≈ 0.26) en biedt dus een compacte, goed-verklarende beschrijving van de determinanten van studiesucces.

***

(e) Welke variabelen hebben uiteindelijk een significante invloed?

| Variabele       | Richting | p-waarde | Odds Ratio | Interpretatie                                                                                                               |
| --------------- | -------- | -------- | ---------- | --------------------------------------------------------------------------------------------------------------------------- |
| **log(Apps)**   | Positief | 0.000    | 1.79       | Universiteiten met meer aanmeldingen hebben een hogere kans op een hoog slagingssucces.                                     |
| **Outstate**    | Positief | 0.000    | 2.44       | Hogere out-of-state collegegelden (meer selectieve, duurdere instellingen) vergroten de kans op hoog slagingssucces.        |
| **Top10perc**   | Positief | 0.000    | 2.11       | Een groter aandeel top-10 % studenten vergroot de kans op hoog slagingssucces.                                              |
| **Private_bin** | Positief | 0.002    | 2.99       | Private universiteiten hebben gemiddeld bijna drie keer zo grote kans op een hoog slagingssucces als publieke instellingen. |



## Conclusie:
Uiteindelijk blijken vier variabelen een significante positieve invloed te hebben op de kans dat een universiteit een hoog slagingssucces behaalt:
log(Apps) – Meer aanmeldingen → grotere kans op succes.
Grotere en populairdere instellingen trekken sterkere studenten aan en kunnen selectiever zijn.
Outstate – Hogere collegegelden → hogere kans op succes.
Duidt op financiële draagkracht en selectiviteit van de instelling.
Top10perc – Hogere instroomkwaliteit → hogere kans op succes.
Universiteiten met meer top-10 % studenten presteren duidelijk beter.
Private_bin – Private universiteiten → hogere kans op succes.
Waarschijnlijk door kleinere klassen, intensievere begeleiding en strengere toelatingsnormen.


***

(f) Bereken het percentage goed voorspelde scholen zowel voor de estimation sample
als voor de test sample.

We berekenen voor zowel de *estimation sample* (trainingsset) als de *test sample* het percentage
correct geclassificeerde scholen (accuracy).  
Hiervoor gebruiken we het eindmodel met de variabelen:  
**log(Apps)**, **Outstate**, **Grad_Rate**, en **PhD**.


In [14]:
# === 6(f) Percentage goed voorspelde scholen — kolommen alignen met final_res ===



feature_cols=['log_apps', 'Outstate', 'Top10perc', 'Private_bin']

# Bouw prediction matrices met exact die kolommen
#    - Train: we hebben al X_train_imp; voeg constant toe en reindex
X_train_pred = sm.add_constant(X_train_imp[feature_cols], has_constant='add')
X_train_pred = X_train_pred.reindex(columns=final_cols, fill_value=0)

# - Test: idem
X_test_pred = sm.add_constant(X_test_imp[feature_cols], has_constant='add')
X_test_pred = X_test_pred.reindex(columns=final_cols, fill_value=0)


# Voorspel probabiliteiten en klassen (drempel 0.5)
y_train_prob = final_res.predict(X_train_pred)
y_test_prob  = final_res.predict(X_test_pred)

y_train_pred = (y_train_prob >= 0.5).astype(int)
y_test_pred  = (y_test_prob  >= 0.5).astype(int)


acc_train = accuracy_score(y_train, y_train_pred)
acc_test  = accuracy_score(y_test,  y_test_pred)
auc_train = roc_auc_score(y_train,  y_train_prob)
auc_test  = roc_auc_score(y_test,   y_test_prob)

print("=== 6(f) Modelprestatie ===")
print(f"Accuracy (estimation/train): {acc_train:.3f}")
print(f"Accuracy (test):             {acc_test:.3f}")
print(f"AUC (train):                 {auc_train:.3f}")
print(f"AUC (test):                  {auc_test:.3f}")

print("\nConfusion Matrix (train):")
print(confusion_matrix(y_train, y_train_pred))

print("\nConfusion Matrix (test):")
print(confusion_matrix(y_test, y_test_pred))

print("\nClassification Report (test):")
print(classification_report(y_test, y_test_pred, digits=3))


=== 6(f) Modelprestatie ===
Accuracy (estimation/train): 0.756
Accuracy (test):             0.759
AUC (train):                 0.831
AUC (test):                  0.869

Confusion Matrix (train):
[[149  78]
 [ 64 291]]

Confusion Matrix (test):
[[ 45  31]
 [ 16 103]]

Classification Report (test):
              precision    recall  f1-score   support

           0      0.738     0.592     0.657        76
           1      0.769     0.866     0.814       119

    accuracy                          0.759       195
   macro avg      0.753     0.729     0.736       195
weighted avg      0.757     0.759     0.753       195



## Resultaten:

Percentage goed voorspeld (Accuracy)
- **Estimation (train):** 75.6 %
- **Test:** 75.9 %

Discriminatie (AUC)
- **Train AUC:** 0.831  
- **Test AUC:** 0.869  

Test Confusion Matrix (threshold = 0.5)
| | Voorspeld = 0 | Voorspeld = 1 |
|:--|:--:|:--:|
| **Werkelijk = 0** | TN = 45 | FP = 31 |
| **Werkelijk = 1** | FN = 16 | TP = 103 |

**Recall (sensitiviteit):**
- Klasse 1 (hoog slagingssucces): **0.866**  
- Klasse 0 (geen hoog slagingssucces): **0.592**

Het model herkent **hogere slagingssuccessen beter** dan lage (hogere sensitiviteit voor 1, lagere specificiteit voor 0).  


## Conclusie:
Het compacte logistische model uit 6(d) voorspelt ~76% van de scholen correct in zowel de estimation- als test-sample. De AUC ≈ 0.87 op de testset duidt op goede discriminatie tussen scholen met laag vs. hoog slagingssucces. Het model heeft hoge gevoeligheid voor succes (1) maar is strenger voor niet-succes (0); desgewenst kan de drempel worden aangepast om dit te balanceren.



TP = 103  # True Positives
TN = 45   # True Negatives
FP = 31   # False Positives
FN = 16   # False Negatives
TP = 103  # True Positives


Accuracy = (TP + TN) / (TP+TN+FP+FN)
Recall (sensitiviteit) = TP / (TP + FN)
Specificity = TN / (TN + FP)
Precision = TP / (TP + FP)
F1 = 2 * (precision * recall) / (precision + recall)
#Voor de in je notebook genoemde test-matrix (TN=45, FP=31, FN=16, TP=103):

Tot = 195
Accuracy ≈ 148/195 ≈ 0.759

Recall (klasse 1) ≈ 103/119 ≈ 0.866
Specificity ≈ 45/76 ≈ 0.592
Precision ≈ 103/134 ≈ 0.769
F1 ≈ 0.813
 